In [1]:
import importlib
import sys

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')
sys.path.insert(0, '../../../../../..')

In [2]:
# Load decision-labeled data
file_path_train = '../../../../../../data/Sepsis/tensor_data/decision_labeled/sepsis_all_5_train.pkl'
sepsis_train_dataset = torch.load(file_path_train, weights_only=False)

file_path_val = '../../../../../../data/Sepsis/tensor_data/decision_labeled/sepsis_all_5_val.pkl'
sepsis_val_dataset = torch.load(file_path_val, weights_only=False)

In [3]:
# Sepsis dataset dynamic categories, features:
sepsis_all_categories = sepsis_train_dataset.all_categories

sepsis_all_categories_cat = sepsis_all_categories[0]
# print(sepsis_all_categories_cat)
sepsis_all_categories_num = sepsis_all_categories[1]
# print(sepsis_all_categories_num)
for i, cat in enumerate(sepsis_all_categories_cat):
     print(f"Sepsis dynamic categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(sepsis_all_categories_num):
     print(f"Sepsis dynamic numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of numerical: {num[1]}")
print('\n')
     
# Sepsis dataset static categories, features:
sepsis_all_stat_categories = sepsis_train_dataset.all_static_categories

sepsis_all_stat_categories_cat = sepsis_all_stat_categories[0]
# print(sepsis_all_stat_categories_cat)
sepsis_all_stat_categories_num = sepsis_all_stat_categories[1]
# print(sepsis_all_stat_categories_num)
for i, cat in enumerate(sepsis_all_stat_categories_cat):
     print(f"Sepsis static categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(sepsis_all_stat_categories_num):
     print(f"Sepsis static numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of numerical: {num[1]}")  

Sepsis dynamic categorical feature: concept:name, Index position in categorical data list: 0
Sepsis amount of category labels: 18
Sepsis dynamic categorical feature: org:group, Index position in categorical data list: 1
Sepsis amount of category labels: 28
Sepsis dynamic categorical feature: lifecycle:transition, Index position in categorical data list: 2
Sepsis amount of category labels: 3


Sepsis dynamic numerical feature: case_elapsed_time, Index position in categorical data list: 0
Sepsis amount of numerical: 1
Sepsis dynamic numerical feature: event_elapsed_time, Index position in categorical data list: 1
Sepsis amount of numerical: 1
Sepsis dynamic numerical feature: day_in_week, Index position in categorical data list: 2
Sepsis amount of numerical: 1
Sepsis dynamic numerical feature: seconds_in_day, Index position in categorical data list: 3
Sepsis amount of numerical: 1
Sepsis dynamic numerical feature: Leucocytes, Index position in categorical data list: 4
Sepsis amount of nu

In [4]:
# Create lists with name of Encoder features (input) and decoder features (input & output)
# Encoder features:
enc_feat_cat = []
enc_feat_num = []
for cat in sepsis_all_categories_cat:
    enc_feat_cat.append(cat[0])
for num in sepsis_all_categories_num:
    enc_feat_num.append(num[0])
enc_feat = [enc_feat_cat, enc_feat_num]
print("Input features encoder: ", enc_feat)

# Static encoder features:
stat_enc_feat_cat = []
stat_enc_feat_num = []
for cat in sepsis_all_stat_categories_cat:
     stat_enc_feat_cat.append(cat[0])
for num in sepsis_all_stat_categories_num:
     stat_enc_feat_num.append(num[0])
stat_enc_feat = [stat_enc_feat_cat, stat_enc_feat_num]
print("Input features static encoder: ", stat_enc_feat)

# Decoder features:
dec_feat_cat = ['concept:name']
dec_feat_num = []
dec_feat = [dec_feat_cat, dec_feat_num]
print("Features decoder: ", dec_feat)

# Guard tensors are pre-computed and stored in the dataset .pkl files
# (prepared during data loading via prepare_guard_tensors).
print(f"Guard tensors: targets {sepsis_train_dataset._guard_targets.shape}, "
      f"mask {sepsis_train_dataset._guard_mask.shape}, "
      f"confidence {sepsis_train_dataset._guard_confidence.shape}")

Input features encoder:  [['concept:name', 'org:group', 'lifecycle:transition'], ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_day', 'Leucocytes', 'CRP', 'LacticAcid']]
Input features static encoder:  [['Age', 'InfectionSuspected', 'Diagnose', 'DiagnosticLacticAcid', 'DiagnosticBlood', 'DiagnosticArtAstrup', 'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor', 'DiagnosticOther', 'DiagnosticUrinarySediment', 'DiagnosticECG', 'DiagnosticUrinaryCulture', 'DiagnosticXthorax', 'SIRSCritTachypnea', 'SIRSCritHeartRate', 'SIRSCriteria2OrMore', 'SIRSCritTemperature', 'SIRSCritLeucos', 'Hypotensie', 'Oligurie', 'Infusion', 'Hypoxie', 'DisfuncOrg'], []]
Features decoder:  [['concept:name'], []]
Guard tensors: targets torch.Size([9750, 50, 18]), mask torch.Size([9750, 50]), confidence torch.Size([9750, 50])


In [5]:
import suffix_pred.models.K_UED_LSTM
importlib.reload(suffix_pred.models.K_UED_LSTM)
from suffix_pred.models.K_UED_LSTM import DropoutUncertaintyEncoderDecoderLSTM

# Prediction decoder output sequence length
seq_len_pred = sepsis_train_dataset.min_suffix_size

# Size hidden layer
hidden_size = 128

# Number of cells
num_layers = 4

# Fixed Dropout probability 
# Experiment with different values but be carefull with exploiting gradients:
dropout = 0.1

# Encoder Decoder model initialization
model = DropoutUncertaintyEncoderDecoderLSTM(data_set_categories=sepsis_all_categories,
                                             enc_feat=enc_feat,
                                             dec_feat=dec_feat,
                                             seq_len_pred=seq_len_pred,
                                             hidden_size=hidden_size,
                                             num_layers=num_layers,
                                             dropout=dropout,
                                             # static attributes
                                             static_data_set_categories=sepsis_all_stat_categories,
                                             static_enc_feat=stat_enc_feat
                                             )

Dynamic data set categories:  ([('concept:name', 18, {'Admission IC': 1, 'Admission NC': 2, 'CRP': 3, 'EOS': 4, 'ER Registration': 5, 'ER Sepsis Triage': 6, 'ER Triage': 7, 'IV Antibiotics': 8, 'IV Liquid': 9, 'LacticAcid': 10, 'Leucocytes': 11, 'Release A': 12, 'Release B': 13, 'Release C': 14, 'Release D': 15, 'Release E': 16, 'Return ER': 17}), ('org:group', 28, {'?': 1, 'A': 2, 'B': 3, 'C': 4, 'D': 5, 'E': 6, 'EOS': 7, 'F': 8, 'G': 9, 'H': 10, 'I': 11, 'J': 12, 'K': 13, 'L': 14, 'M': 15, 'N': 16, 'O': 17, 'P': 18, 'Q': 19, 'R': 20, 'S': 21, 'T': 22, 'U': 23, 'V': 24, 'W': 25, 'X': 26, 'Y': 27}), ('lifecycle:transition', 3, {'EOS': 1, 'complete': 2})], [('case_elapsed_time', 1, {}), ('event_elapsed_time', 1, {}), ('day_in_week', 1, {}), ('seconds_in_day', 1, {}), ('Leucocytes', 1, {}), ('CRP', 1, {}), ('LacticAcid', 1, {})])
Data set static categories:  ([('Age', 16, {'20': 1, '25': 2, '30': 3, '35': 4, '40': 5, '45': 6, '50': 7, '55': 8, '60': 9, '65': 10, '70': 11, '75': 12, '80':

In [6]:
import suffix_pred.loss
importlib.reload(suffix_pred.loss)
from suffix_pred.loss import Loss

loss_obj = Loss()

In [ ]:
import suffix_pred.train
importlib.reload(suffix_pred.train)
from suffix_pred.train import UEDTrainer

from torch.optim.lr_scheduler import ReduceLROnPlateau

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Start learning rate
learning_rate = 1e-5
weight_decay = 0.0

# Optimizer and Scheduler
optimizer = torch.optim.AdamW(params=model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=15, min_lr=1e-8)


# Keep consistent across all models
# Epochs / Batch size
num_epochs = 100
batch_size = 128

# L2 regularization term
regularization_term = 1e-4

# Shuffle data
shuffle = True

# Scheduled sampling (same as clean baseline):
# TF ratio anneals from 1.0 → min_teacher_forcing_value via inverse-sigmoid.
# L_guard is applied only at decoder steps that consumed GT input (tf_mask=1).
teacher_forcing_mode = "scheduled"
max_teacher_forcing_value = 1.0
min_teacher_forcing_value = 0.0

optimize_values = {"regularization_term": regularization_term,
                   "optimizer": optimizer,
                   "scheduler": scheduler,
                   "epochs": num_epochs,
                   "mini_batches": batch_size,
                   "shuffle": shuffle,
                   "min_teacher_forcing_value": min_teacher_forcing_value,
                   "max_teacher_forcing_value": max_teacher_forcing_value,
                   "teacher_forcing_mode": teacher_forcing_mode}

required_optimize_keys = {"regularization_term",
                          "optimizer",
                          "scheduler",
                          "epochs",
                          "mini_batches",
                          "shuffle",
                          "min_teacher_forcing_value",
                          "max_teacher_forcing_value",
                          "teacher_forcing_mode"}

missing_keys = required_optimize_keys.difference(optimize_values.keys())
if missing_keys:
    raise ValueError(f"Missing required optimize_values keys: {sorted(missing_keys)}")

suffix_data_split_value = sepsis_train_dataset.min_suffix_size
print("Train seq length:", suffix_data_split_value)
print(f"Teacher forcing config: mode={teacher_forcing_mode}")

# Decision-aware guard regularization weight (λ_g)
lambda_g = 1.0
print(f"Decision-aware training: λ_g = {lambda_g}")

model_output_path = "../../../../../../models/Sepsis/decision/Sepsis_UED_LSTM_v1_DA.pkl"
os.makedirs(os.path.dirname(model_output_path), exist_ok=True)

trainer = UEDTrainer(device=device,
                     model=model,
                     data_train=sepsis_train_dataset,
                     data_val=sepsis_val_dataset,
                     loss_obj=loss_obj,
                     log_normal_loss_num_feature=[],
                     optimize_values=optimize_values,
                     suffix_data_split_value=suffix_data_split_value,
                     lambda_g=lambda_g,
                     save_model_n_th_epoch=1,
                     saving_path=model_output_path)

# Train the model (decision-aware: L_base + λ_g * L_guard)
train_attenuated_losses, val_losses, val_attenuated_losses = trainer.train_model(use_statics=True,
                                                                                 use_zero_padd_masking=True,
                                                                                 use_eos_padd_masking=True)

Device: cuda


Train seq length: 5
Teacher forcing config: mode=scheduled
Decision-aware training: λ_g = 1.0


Device:  cuda
Model:  DropoutUncertaintyEncoderDecoderLSTM(
  (embeddings_enc): ModuleList(
    (0): Embedding(18, 8)
    (1): Embedding(28, 10)
    (2): Embedding(3, 3)
  )
  (embeddings_static_enc): ModuleList(
    (0): Embedding(16, 8)
    (1): Embedding(3, 3)
    (2): Embedding(108, 22)
    (3-23): 21 x Embedding(3, 3)
  )
  (encoder): DropoutUncertaintyLSTMEncoder(
    (embeddings): ModuleList(
      (0): Embedding(18, 8)
      (1): Embedding(28, 10)
      (2): Embedding(3, 3)
    )
    (static_embeddings): ModuleList(
      (0): Embedding(16, 8)
      (1): Embedding(3, 3)
      (2): Embedding(108, 22)
      (3-23): 21 x Embedding(3, 3)
    )
    (input_proj): Linear(in_features=124, out_features=128, bias=True)
    (layernorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (act): ReLU()
    (first_layer): DropoutUncertaintyLSTMCell(
      (Wi): Linear(in_features=128, out_features=128, bias=True)
      (Ui): Linear(in_features=128, out_features=128, bias=True)
      (

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [1/100], Learning Rate: 1e-05, Teacher forcing ratio: 1.0000, Scheduled sampling epsilon: 0.0000
Training: Avg Attenuated Training Loss: 5.2550


...

Validation: Avg Standard Validation Loss: 1.6849
Validation: Avg Attenuated Validation Loss: 1.6926
Validation Loss for Scheduler: 1.6849
saving model


Epoch [99/100], Learning Rate: 1e-05, Teacher forcing ratio: 0.0006, Scheduled sampling epsilon: 0.9994
Training: Avg Attenuated Training Loss: 2.3910


Validation: Avg Standard Validation Loss: 1.6847
Validation: Avg Attenuated Validation Loss: 1.6922
Validation Loss for Scheduler: 1.6847
saving model


Epoch [100/100], Learning Rate: 1e-05, Teacher forcing ratio: 0.0006, Scheduled sampling epsilon: 0.9994
Training: Avg Attenuated Training Loss: 2.3918


Validation: Avg Standard Validation Loss: 1.6802
Validation: Avg Attenuated Validation Loss: 1.6876
Validation Loss for Scheduler: 1.6802
saving model
Training complete.
Model saved to path: ../../../../../../models/Sepsis/decision/Sepsis_UED_LSTM_v2_DA.pkl
